## Semantic Query Caching Simulation — Wikidata (`fig:caching`)

Wikidata twin of `caching_simulation_dbpedia.ipynb`, implementing the same
protocol from `notebooks/caching/README.md` (§Downstream Application in the
paper):

- **Stream:** held-out SQuARE **test split** anchors, seeded shuffle
  (1,074 Wikidata test anchors, vs. 472 on DBpedia).
- **Cache:** FIFO over the **K most recent** executed queries (embedding +
  result set). Hit when max cosine ≥ θ → return the cached result, do not
  insert. Miss → "execute" (oracle lookup) and insert.
- **Grid:** K ∈ {50, 100, 200} × θ ∈ {0.7, 0.8, 0.9}.
- **Models:** SparqlCE, TF-IDF, BM25, E5-base-v2, Qwen2.5-Coder-0.5B
  (generative code LLM, last-token), CodeBERT, GraphCodeBERT, SPBERT,
  SBERT/MiniLM, SBERT/MPNet, and the fine-tuned E5-base-v2 baseline.
- **Metrics:** hit rate; result quality = mean answer-set Jaccard between the
  returned (cached) result and the true result, over hits.

**Before running:**

- SparqlCE checkpoint: `sparqlce/models/wikidata_model_mrr_0.3238.pt`
  (not in git — see the main README).
- Fine-tuned E5-base-v2: `checkpoints/e5_square_wikidata/best_model`
  (produced by `notebooks/finetuning/05_e5_contrastive_finetune_square_wikidata.ipynb`).
- `pip install ijson` — `datasets/result_sets_wikidata.json` is ~6GB (vs.
  ~4MB for DBpedia: Wikidata answer sets run far longer per query), so this
  notebook streams and filters it instead of loading it whole. See the data
  cell below for why.

Everything else needed is already in `datasets/`. Outputs
`results/caching/caching_wikidata.json` plus the same headline-table and
utility numbers reported for DBpedia's Figure 4, so the two graphs stay
directly comparable.

In [ ]:
import gc
import os

# Must be set before the first CUDA call (i.e. before `import torch` touches
# the GPU), so cuBLAS allocates a deterministic workspace.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
import json
import random
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# ---- Project root (works from notebooks/caching/ or anywhere in the repo) ----
_PROJECT_ROOT = Path.cwd()
while not (_PROJECT_ROOT / "datasets").exists():
    assert _PROJECT_ROOT != _PROJECT_ROOT.parent, "could not locate repo root"
    _PROJECT_ROOT = _PROJECT_ROOT.parent

if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))


# ---- Config ----
TRIPLETS_PATH = (
    _PROJECT_ROOT
    / "datasets"
    / "wikidata-triplets-dataset.json"
)

SPLITS_PATH = (
    _PROJECT_ROOT
    / "datasets"
    / "splits_wikidata.json"
)

RESULTS_PATH = (
    _PROJECT_ROOT
    / "datasets"
    / "result_sets_wikidata.json"
)

SPARQLCE_CKPT = (
    _PROJECT_ROOT
    / "sparqlce"
    / "models"
    / "wikidata_model_mrr_0.3238.pt"
)

OUT_DIR = (
    _PROJECT_ROOT
    / "results"
    / "caching"
)


SEED = 42

K_GRID = [
    50,
    100,
    200,
]

THETA_GRID = [
    0.7,
    0.8,
    0.9,
]

# (K, theta) row reported in the main table -- same operating point as the
# DBpedia run, so the two graphs stay directly comparable.
HEADLINE = (100, 0.8)


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ---------------------------------------------------------------------------
# Determinism.
#
# SEED only fixes the stream shuffle order and RNG-based initialization -- it
# does not make CUDA kernels bit-reproducible run to run. On the DBpedia run,
# repeated executions showed SparqlCE (GATv2 scatter/gather message passing)
# and both Qwen3 paths (dense attention kernels) drifting between runs, while
# every sparse/SentenceTransformer baseline (TF-IDF, BM25, E5, MiniLM, MPNet)
# stayed bit-identical across reruns. Force determinism everywhere torch
# supports it; an op with no deterministic GPU kernel will raise here instead
# of silently drifting into a different cache trajectory. See the SparqlCE
# encode cell below for a CPU fallback if the GNN's aggregation still raises.
# ---------------------------------------------------------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

print(
    f"project root: {_PROJECT_ROOT}\n"
    f"device: {DEVICE}"
)

In [ ]:
def normalize_query(q: str) -> str:
    """
    Normalize query formatting for exact oracle lookup only.

    The original query string is still used for embedding.
    """
    return " ".join(q.split())


def freeze_json(value):
    """
    Convert a JSON-compatible value into a recursively hashable form.

    This allows result rows represented as lists or dictionaries to be
    stored safely in a frozenset for answer-set Jaccard computation.
    """
    if isinstance(value, dict):
        return tuple(
            sorted(
                (
                    key,
                    freeze_json(item),
                )
                for key, item in value.items()
            )
        )

    if isinstance(value, list):
        return tuple(
            freeze_json(item)
            for item in value
        )

    return value


with open(
    TRIPLETS_PATH,
    encoding="utf-8",
) as f:
    triplets = json.load(f)


with open(
    SPLITS_PATH,
    encoding="utf-8",
) as f:
    splits = json.load(f)


# ---------------------------------------------------------------------------
# Build train/test anchor collections from the committed split, BEFORE
# touching the oracle file: the stream-filter below needs to know which
# queries matter first.
# ---------------------------------------------------------------------------
test_anchors = [
    triplets[i]["anchor_query"]
    for i in splits["test_idx"]
]

train_anchors = [
    triplets[i]["anchor_query"]
    for i in splits["train_idx"]
]


# ---------------------------------------------------------------------------
# Stream-filter the oracle result sets.
#
# result_sets_wikidata.json is ~6GB (vs. ~4MB for DBpedia: Wikidata answer
# sets run far longer per query). A plain json.load would parse the whole
# file -- including the ~8,600 train-side queries the cache simulation never
# looks up -- and hold it all in memory at once. ijson streams key/value
# pairs one at a time, so only the ~1,074 test-stream queries are ever kept.
#
# Query keys are normalized only for exact oracle lookup. Result rows are
# converted into hashable canonical Python structures. A normalization
# collision with different result sets is treated as an error rather than
# silently overwriting one oracle entry with another.
# ---------------------------------------------------------------------------
try:
    import ijson
except ImportError as exc:
    raise ImportError(
        "This notebook requires `ijson` to stream the ~6GB Wikidata "
        "result-sets file without loading it whole into memory. Install it "
        "with `pip install ijson` and re-run this cell."
    ) from exc

needed_keys = {
    normalize_query(q)
    for q in test_anchors
}

result_sets = {}

with open(RESULTS_PATH, "rb") as f:
    for raw_query, rows in tqdm(
        ijson.kvitems(f, ""),
        desc="scanning oracle result sets",
    ):

        if not rows:
            continue

        query_key = normalize_query(raw_query)

        if query_key not in needed_keys:
            continue

        answer_set = frozenset(
            freeze_json(row)
            for row in rows
        )

        if (
            query_key in result_sets
            and result_sets[query_key] != answer_set
        ):
            raise ValueError(
                "Conflicting oracle result sets after query normalization:\n"
                f"{query_key}"
            )

        result_sets[query_key] = answer_set


# ---------------------------------------------------------------------------
# Keep ORIGINAL query strings in the stream for embedding.
#
# Normalized query strings are used only for oracle lookup.
# ---------------------------------------------------------------------------
stream = [
    q
    for q in test_anchors
    if normalize_query(q) in result_sets
]

dropped = (
    len(test_anchors)
    - len(stream)
)


# ---------------------------------------------------------------------------
# Shuffle the occurrence-level stream deterministically.
#
# Important:
# This deliberately does NOT deduplicate repeated anchors. If the same query
# occurs multiple times in the held-out workload, those occurrences remain
# separate stream events.
# ---------------------------------------------------------------------------
random.seed(SEED)
random.shuffle(stream)


# ---------------------------------------------------------------------------
# Diagnostics only. These values do not change the experiment.
# ---------------------------------------------------------------------------
stream_unique_queries = len(
    {
        normalize_query(q)
        for q in stream
    }
)

duplicate_stream_events = (
    len(stream)
    - stream_unique_queries
)


print(
    f"test anchors: {len(test_anchors)} | "
    f"in stream: {len(stream)} | "
    f"dropped (missing/empty oracle result): {dropped}"
)

print(
    f"unique normalized queries in stream: "
    f"{stream_unique_queries} | "
    f"repeated stream events: "
    f"{duplicate_stream_events}"
)


def jaccard(
    a: frozenset,
    b: frozenset,
) -> float:
    """
    Answer-set Jaccard similarity.
    """
    union = len(a | b)

    return (
        len(a & b) / union
        if union
        else 0.0
    )

In [ ]:
# ============================================================================
# 2. Embed the stream once per model
#
# Every representation is L2-normalised so that the cache lookup can continue
# to use a common cosine-similarity threshold grid.
# ============================================================================

from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

stream_embeddings = {}


def clear_model(model):
    """Release a transformer model and clear accelerator memory."""
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# SparqlCE — trained checkpoint
#
# GATv2's scatter/gather message passing is the one op in this notebook that
# may lack a deterministic CUDA kernel under torch.use_deterministic_algorithms.
# Try the default device first; if that op raises, fall back to CPU (cheap --
# only ~1,000 short queries) rather than silently accept a nondeterministic
# run.
# ---------------------------------------------------------------------------
from sparqlce.embedder import SPARQLEmbedder
from sparqlce.config import Config as _SparqlCEConfig


def _encode_with_sparqlce(stream, batch_size=32):
    original_device = _SparqlCEConfig.DEVICE
    try:
        embedder = SPARQLEmbedder(str(SPARQLCE_CKPT))
        embs = [
            embedder.encode(stream[i : i + batch_size])
            for i in tqdm(
                range(0, len(stream), batch_size), desc="SparqlCE"
            )
        ]
    except RuntimeError as exc:
        if "deterministic" not in str(exc).lower():
            raise
        print(
            f"SparqlCE GPU encode is non-deterministic ({exc});"
            " falling back to CPU."
        )
        _SparqlCEConfig.DEVICE = torch.device("cpu")
        embedder = SPARQLEmbedder(str(SPARQLCE_CKPT))
        embs = [
            embedder.encode(stream[i : i + batch_size])
            for i in tqdm(
                range(0, len(stream), batch_size), desc="SparqlCE (CPU)"
            )
        ]
    finally:
        _SparqlCEConfig.DEVICE = original_device
    return embedder, embs


sparqlce, embs = _encode_with_sparqlce(stream)

stream_embeddings["SparqlCE"] = normalize(
    np.vstack(embs),
    norm="l2",
    axis=1,
).astype(
    np.float32,
    copy=False,
)

del sparqlce
del embs
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# TF-IDF
#
# Vocabulary and IDF statistics are fit on TRAIN anchors only.
# ---------------------------------------------------------------------------
tfidf = TfidfVectorizer(
    norm="l2",
).fit(
    train_anchors
)

stream_embeddings["TF-IDF"] = (
    tfidf
    .transform(stream)
    .toarray()
    .astype(np.float32, copy=False)
)


# ---------------------------------------------------------------------------
# BM25
#
# The cache simulator requires a bounded cosine similarity because theta is
# shared across all baselines. We therefore build Okapi-BM25 term-weight
# vectors using TRAIN-anchor corpus statistics, then L2-normalise each vector.
# Cache lookup remains a cosine comparison between query representations.
# ---------------------------------------------------------------------------
class BM25Vectorizer:
    """Okapi-BM25 term weighting with optional row normalisation."""

    def __init__(
        self,
        *,
        k1=1.5,
        b=0.75,
        norm="l2",
        **count_vectorizer_kwargs,
    ):
        self.k1 = float(k1)
        self.b = float(b)
        self.norm = norm
        self.vectorizer = CountVectorizer(
            **count_vectorizer_kwargs
        )

    def fit(self, documents):
        counts = (
            self.vectorizer
            .fit_transform(documents)
            .tocsr()
            .astype(np.float32)
        )

        n_documents = counts.shape[0]

        if n_documents == 0:
            raise ValueError(
                "BM25Vectorizer cannot be fit on an empty corpus."
            )

        document_frequency = np.asarray(
            (counts > 0).sum(axis=0)
        ).ravel()

        # Robertson/Sparck Jones IDF with the common +1 stabilisation.
        self.idf_ = np.log(
            1.0
            + (
                n_documents
                - document_frequency
                + 0.5
            )
            / (
                document_frequency
                + 0.5
            )
        ).astype(np.float32)

        document_lengths = np.asarray(
            counts.sum(axis=1)
        ).ravel()

        self.avg_document_length_ = float(
            document_lengths.mean()
        )

        if self.avg_document_length_ <= 0:
            raise ValueError(
                "BM25Vectorizer received only empty documents."
            )

        return self

    def transform(self, documents):
        counts = (
            self.vectorizer
            .transform(documents)
            .tocoo()
            .astype(np.float32)
        )

        document_lengths = np.asarray(
            counts.tocsr().sum(axis=1)
        ).ravel()

        length_factor = (
            1.0
            - self.b
            + self.b
            * (
                document_lengths
                / self.avg_document_length_
            )
        )

        denominator = (
            counts.data
            + self.k1
            * length_factor[counts.row]
        )

        weighted_data = (
            self.idf_[counts.col]
            * counts.data
            * (self.k1 + 1.0)
            / denominator
        )

        weighted = sparse.csr_matrix(
            (
                weighted_data,
                (
                    counts.row,
                    counts.col,
                ),
            ),
            shape=counts.shape,
            dtype=np.float32,
        )

        if self.norm is not None:
            weighted = normalize(
                weighted,
                norm=self.norm,
                axis=1,
                copy=False,
            )

        return weighted

    def fit_transform(self, documents):
        return (
            self
            .fit(documents)
            .transform(documents)
        )


bm25 = BM25Vectorizer(
    k1=1.5,
    b=0.75,
    norm="l2",
).fit(
    train_anchors
)

stream_embeddings["BM25"] = (
    bm25
    .transform(stream)
    .toarray()
    .astype(np.float32, copy=False)
)


# ---------------------------------------------------------------------------
# Dense text-embedding baselines
# ---------------------------------------------------------------------------
model_specs = [
    {
        "name": "E5-base-v2",
        "model_id": "intfloat/e5-base-v2",
        "texts": [
            f"query: {q}"
            for q in stream
        ],
        "batch_size": 32,
    },
    {
        "name": "SBERT/MiniLM",
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "texts": stream,
        "batch_size": 64,
    },
    {
        "name": "SBERT/MPNet",
        "model_id": "sentence-transformers/all-mpnet-base-v2",
        "texts": stream,
        "batch_size": 32,
    },
    {
        "name": "CodeBERT",
        "model_id": "microsoft/codebert-base",
        "texts": stream,
        "batch_size": 64,
    },
    {
        "name": "GraphCodeBERT",
        "model_id": "microsoft/graphcodebert-base",
        "texts": stream,
        "batch_size": 64,
    },
    {
        "name": "SPBERT",
        "model_id": "razent/spbert-mlm-wso-base",
        "texts": stream,
        "batch_size": 64,
    },
]


for spec in model_specs:
    model = SentenceTransformer(
        spec["model_id"],
        device=DEVICE,
    )

    encoded = model.encode(
        spec["texts"],
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=True,
        batch_size=spec["batch_size"],
    ).astype(
        np.float32,
        copy=False,
    )

    # Explicit post-normalisation avoids small model/dtype-dependent norm drift
    # (observed for Qwen3-Embedding-0.6B despite normalize_embeddings=True).
    stream_embeddings[
        spec["name"]
    ] = normalize(
        encoded,
        norm="l2",
        axis=1,
        copy=False,
    ).astype(
        np.float32,
        copy=False,
    )

    del encoded
    clear_model(model)


# ============================================================================
# Qwen2.5-Coder-0.5B — generative code LLM used as an embedder (last-token
# pooling), replacing the dense Qwen3-Embedding-0.6B. Off-label use of a
# generative model; 0.5B roughly scale-matches the 0.6B it replaces.
# ============================================================================
from transformers import AutoModel, AutoTokenizer
import torch.nn.functional as F

CODER_MODEL = "Qwen/Qwen2.5-Coder-0.5B"
CODER_NAME = "Qwen2.5-Coder-0.5B"

def _last_token_pool(h, am):
    if (am[:, -1].sum() == am.shape[0]):        # left-padded
        return h[:, -1]
    lens = am.sum(dim=1) - 1
    idx = torch.arange(h.shape[0], device=h.device)
    return h[idx, lens]

coder_tok = AutoTokenizer.from_pretrained(CODER_MODEL, padding_side="left")
if coder_tok.pad_token_id is None:
    coder_tok.pad_token = coder_tok.eos_token
coder_model = AutoModel.from_pretrained(CODER_MODEL, torch_dtype=torch.float32).to(DEVICE).eval()

@torch.inference_mode()
def encode_coder(queries, batch_size=8, max_length=512):
    out = []
    for start in tqdm(range(0, len(queries), batch_size), desc=CODER_NAME):
        batch = queries[start:start + batch_size]
        tok = coder_tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        h = coder_model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"]).last_hidden_state
        emb = F.normalize(_last_token_pool(h, tok["attention_mask"]).float(), p=2, dim=1)
        out.append(emb.cpu().numpy())
    return np.vstack(out).astype(np.float32, copy=False)

stream_embeddings[CODER_NAME] = encode_coder(stream)
del coder_model, coder_tok
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================================
# Fine-tuned E5-base-v2 — SQuARE Wikidata full fine-tune (capacity-matched
# fine-tuned baseline; replaces the fine-tuned Qwen3 baseline). Mean pooling,
# "query: " prefix, matching 05_e5_contrastive_finetune_square_wikidata.ipynb.
# ============================================================================
FT_E5_PATH = _PROJECT_ROOT / "checkpoints" / "e5_square_wikidata" / "best_model"
FT_E5_NAME = "E5-base-v2 (fine-tuned)"

if not FT_E5_PATH.exists():
    raise FileNotFoundError(
        f"Fine-tuned E5 model not found: {FT_E5_PATH} — run "
        "notebooks/finetuning/05_e5_contrastive_finetune_square_wikidata.ipynb first."
    )

ft_e5_tok = AutoTokenizer.from_pretrained(str(FT_E5_PATH))
ft_e5_model = AutoModel.from_pretrained(str(FT_E5_PATH), torch_dtype=torch.float32).to(DEVICE).eval()

def _mean_pool(h, am):
    m = am.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.inference_mode()
def encode_ft_e5(queries, batch_size=32, max_length=512):
    out = []
    for start in tqdm(range(0, len(queries), batch_size), desc=FT_E5_NAME):
        batch = [f"query: {q}" for q in queries[start:start + batch_size]]
        tok = ft_e5_tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        h = ft_e5_model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"]).last_hidden_state
        emb = F.normalize(_mean_pool(h, tok["attention_mask"]).float(), p=2, dim=1)
        out.append(emb.cpu().numpy())
    return np.vstack(out).astype(np.float32, copy=False)

stream_embeddings[FT_E5_NAME] = encode_ft_e5(stream)
del ft_e5_model, ft_e5_tok
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Sanity checks
# ---------------------------------------------------------------------------
for name, E in stream_embeddings.items():
    if E.shape[0] != len(stream):
        raise ValueError(
            f"{name}: expected {len(stream)} rows, got {E.shape[0]}"
        )

    row_norms = np.linalg.norm(
        E,
        axis=1,
    )

    nonzero_rows = row_norms > 0

    if np.any(nonzero_rows) and not np.allclose(
        row_norms[nonzero_rows],
        1.0,
        atol=1e-5,
    ):
        raise ValueError(
            f"{name}: non-zero rows are not L2-normalised."
        )

    print(
        f"{name:>24}: {E.shape}"
    )

In [ ]:
# ============================================================================
# 3. Cache simulation
#
# Protocol:
#   - FIFO over the K most recent EXECUTED queries.
#   - Hit when max cosine similarity >= theta.
#   - On a hit: return the cached result and do not modify the cache.
#   - On a miss: use the oracle result and insert the query/result pair.
# ============================================================================

def simulate(
    stream,
    E,
    result_sets,
    K,
    theta,
):
    """
    Run one pass over the query stream.

    E[i] is the L2-normalised embedding of stream[i].
    """

    if len(stream) == 0:
        raise ValueError(
            "Cannot simulate an empty query stream."
        )

    if len(E) != len(stream):
        raise ValueError(
            "Embedding matrix length does not match stream length: "
            f"{len(E)} != {len(stream)}"
        )

    if K <= 0:
        raise ValueError(
            f"K must be positive, got {K}."
        )


    # Each entry is:
    #   (query_embedding, oracle_result_set)
    #
    # maxlen=K gives FIFO eviction automatically.
    cache = deque(
        maxlen=K
    )

    hits = 0
    qualities = []


    for i, q in enumerate(stream):

        R_true = result_sets[
            normalize_query(q)
        ]


        # -------------------------------------------------------------------
        # Cache lookup
        # -------------------------------------------------------------------
        if cache:

            cached_embeddings = np.stack(
                [
                    entry[0]
                    for entry in cache
                ]
            )

            sims = (
                cached_embeddings
                @ E[i]
            )

            best = int(
                np.argmax(sims)
            )


            # ---------------------------------------------------------------
            # Hit:
            #
            # Serve the nearest cached result.
            # Do not execute.
            # Do not insert.
            # Do not refresh cache position.
            # ---------------------------------------------------------------
            if float(sims[best]) >= theta:

                R_cached = cache[best][1]

                hits += 1

                qualities.append(
                    jaccard(
                        R_cached,
                        R_true,
                    )
                )

                continue


        # -------------------------------------------------------------------
        # Miss:
        #
        # "Execute" through the offline oracle and insert into FIFO cache.
        # -------------------------------------------------------------------
        cache.append(
            (
                E[i],
                R_true,
            )
        )


    n_queries = len(stream)

    return {
        "hit_rate": hits / n_queries,
        "result_quality": (
            float(np.mean(qualities))
            if qualities
            else None
        ),
        "n_hits": hits,
        "n_misses": n_queries - hits,
        "n_queries": n_queries,
    }


all_results = {}

for name, E in stream_embeddings.items():

    all_results[name] = {}

    for K in K_GRID:
        for theta in THETA_GRID:

            r = simulate(
                stream=stream,
                E=E,
                result_sets=result_sets,
                K=K,
                theta=theta,
            )

            all_results[name][
                f"K={K},theta={theta}"
            ] = r

    print(
        f"done: {name}"
    )

In [ ]:
# ============================================================================
# 4. Save the full grid + display
# ============================================================================
OUT_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "kg": "wikidata",
    "seed": SEED,
    "stream_length": len(stream),
    "dropped": dropped,
    "grid": {"K": K_GRID, "theta": THETA_GRID},
    "results": all_results,
}
out_json = OUT_DIR / "caching_wikidata.json"
with open(out_json, "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved {out_json}\n")

rows = []
for name, grid in all_results.items():
    for cfg, r in grid.items():
        K, theta = cfg.replace("K=", "").split(",theta=")
        rows.append({"model": name, "K": int(K), "theta": float(theta),
                     "hit_rate": r["hit_rate"], "result_quality": r["result_quality"]})
df = pd.DataFrame(rows)

print("Hit rate:")
print(df.pivot_table(index="model", columns=["K", "theta"], values="hit_rate").round(3))
print("\nResult quality (mean Jaccard over hits):")
print(df.pivot_table(index="model", columns=["K", "theta"], values="result_quality").round(3))

In [ ]:
# ============================================================================
# 5. LaTeX rows + utility numbers for fig:caching (Wikidata twin of Figure 4)
#
# Reports the same "utility" (hit rate x quality, expected served Jaccard per
# query) used in the paper instead of raw hit-rate/quality alone: taken
# separately, a high hit rate with low quality (or vice versa) looks better
# than it is. See src-code/scripts/plot_caching_utility.py for the DBpedia
# figure this mirrors.
# ============================================================================

K_h, theta_h = HEADLINE


order = [
    "SparqlCE",
    "TF-IDF",
    "BM25",
    "E5-base-v2",
    "Qwen2.5-Coder-0.5B",
    "CodeBERT",
    "GraphCodeBERT",
    "SPBERT",
    "E5-base-v2 (fine-tuned)",
    "SBERT/MiniLM",
    "SBERT/MPNet",
]


latex_names = {
    "SparqlCE": r"\textsc{SparqlCE}",
    "TF-IDF": "TF-IDF",
    "BM25": "BM25",
    "E5-base-v2": "E5-base-v2",
    "Qwen2.5-Coder-0.5B": "Qwen2.5-Coder-0.5B",
    "CodeBERT": "CodeBERT",
    "GraphCodeBERT": "GraphCodeBERT",
    "SPBERT": "SPBERT",
    "E5-base-v2 (fine-tuned)": "E5-base-v2 (fine-tuned)",
    "SBERT/MiniLM": "SBERT/MiniLM",
    "SBERT/MPNet": "SBERT/MPNet",
}


def latex_quality(value):
    """
    Format result quality for a LaTeX table.

    Undefined quality occurs when a configuration produces zero hits.
    """
    return (
        f"{value:.3f}"
        if value is not None
        else "--"
    )


def utility(r):
    """hit rate x quality, or None if the config produced no hits."""
    return (
        r["hit_rate"] * r["result_quality"]
        if r["result_quality"] is not None
        else None
    )


print(
    f"% fig:caching (Wikidata) — headline operating point "
    f"(K={K_h}, theta={theta_h}), "
    f"full grid in "
    f"results/caching/caching_wikidata.json"
)


for name in order:

    r = all_results[name][
        f"K={K_h},theta={theta_h}"
    ]

    quality_str = latex_quality(
        r["result_quality"]
    )

    u = utility(r)
    utility_str = f"{u:.3f}" if u is not None else "--"

    print(
        f"{latex_names[name]:<24} "
        f"& $({K_h}, {theta_h})$ "
        f"& {r['hit_rate']:.3f} "
        f"& {quality_str} "
        f"& {utility_str} \\\\"
    )


# ---------------------------------------------------------------------------
# Utility across the theta sweep at K=100, for SparqlCE vs. the fine-tuned
# Qwen3 baseline -- the two panels of Figure 4 on DBpedia.
# ---------------------------------------------------------------------------
print(
    "\n% utility across theta at K=100 (SparqlCE vs. fine-tuned E5):"
)

for theta in THETA_GRID:
    r_sparqlce = all_results["SparqlCE"][f"K=100,theta={theta}"]
    r_e5_ft = all_results["E5-base-v2 (fine-tuned)"][f"K=100,theta={theta}"]

    u_sparqlce = utility(r_sparqlce)
    u_e5_ft = utility(r_e5_ft)

    print(
        f"%   theta={theta}: "
        f"SparqlCE utility={u_sparqlce:.3f} (hit_rate={r_sparqlce['hit_rate']:.3f}), "
        f"E5-ft utility={u_e5_ft:.3f} (hit_rate={r_e5_ft['hit_rate']:.3f})"
    )

### Notes

- BM25 uses L2-normalized Okapi-BM25 term-weight vectors so that it can be
  evaluated with the same cosine-threshold cache protocol as the embedding
  baselines.
- The oracle (`result_sets_wikidata.json`) replaces live endpoint execution,
  so the simulation is deterministic and offline. Only the ~1,074 test-stream
  queries are ever read out of it (see the streaming filter in the data
  cell) -- the file itself is never fully parsed.
- Rerunning with a different stream order: change `SEED` — report the
  default seed-42 numbers alongside the DBpedia results.
- `E5-base-v2 (fine-tuned)` requires the Wikidata E5 checkpoint at
  `checkpoints/e5_square_wikidata/best_model`; the notebook raises a
  clear `FileNotFoundError` if it's missing rather than silently skipping
  the model.